In [1]:
!pip install langchain langchain_community langchain-huggingface pypdf chromadb
!pip install "unstructured[all-docs]"

  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Using cached huggingface_hub-1.4.1-py3-none-any.whl.metadata (13 kB)
Using cached huggingface_hub-1.4.1-py3-none-any.whl (553 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
ERROR: pip's dependency resolver does not currently take into ac

In [2]:
from google.colab import userdata
o = userdata.get('HUGG')

In [3]:
import os
os.environ['HF_TOKEN'] = o

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEndpoint

In [5]:
doc = PyPDFLoader("/content/HR-Policies-Manuals.pdf")
document = doc.load()

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(separators=["\n\n", "\n", " "], chunk_size=500, chunk_overlap=10)
chunks = text_splitter.split_documents(document)

In [7]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings,persist_directory="db")
vectorstore.persist()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipython-input-2020997420.py:5: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [8]:
retriver = vectorstore.as_retriever(search_kwargs = {"k":3})

In [9]:
from langchain_core.runnables import RunnableLambda,RunnableParallel,RunnablePassthrough,RunnableSequence

In [10]:
def f1(data):
  l = []
  for i in data:
    l.append(i.page_content)
  return "\n\n".join(l)

In [11]:
r2 = RunnableLambda(f1)

In [12]:
chain1 = RunnableSequence(retriver,r2)

In [13]:
from langchain_core.prompts import ChatPromptTemplate,HumanMessagePromptTemplate,SystemMessagePromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser

In [14]:
cpt = ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template("""you are a helpful HR who is having 5 years of experience."""),
                                       HumanMessagePromptTemplate.from_template("answer the quention on below context provided context:{context} query:{query}")])

In [15]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

repo_id = "mistralai/Mistral-7B-Instruct-v0.2"

llm = HuggingFaceEndpoint(
    repo_id=repo_id,
    max_new_tokens=512,
    repetition_penalty=1.03,
    huggingfacehub_api_token=os.environ['HF_TOKEN']
)

model = ChatHuggingFace(llm=llm)

In [16]:
otp = StrOutputParser()

In [17]:
chain2 = RunnableParallel({"context": chain1, "query": RunnablePassthrough()})

In [18]:
chain3 = RunnableSequence(chain2,cpt,model,otp)

In [19]:
RAG_pipeline = chain2 | cpt | model | otp

In [20]:
RAG_pipeline.invoke("how many leaves i can get in a month")

"Based on the **leave policy** you provided, an employee's eligibility for leaves in a **month** depends on the **Date of Joining (DoJ)** as follows:\n\n### **Monthly Leave Entitlement (Pro-Rata Basis):**\n| **Date of Joining** | **Number of Leaves per Month** |\n|---------------------|-----------------------------|\n| **1st to 7th**      | **2 leaves**                 |\n| **8th to 14th**     | **1.5 leaves**               |\n| **15th to 21st**    | **1 leave**                   |\n| **22nd to 31st**    | **0.5 leaves**               |\n\n### **Key Notes:**\n1. **If an employee leaves before completing the month** (e.g., joins on **1st June** and leaves on **10th June**), they will **not** get any leave for that month.\n2. **Vailing of leaves** means the remaining unused leaves **carry forward** to the next month but **cannot be combined cumulatively** (only the balance from the previous month can be used).\n3. The **2 leaves/month** apply only to full-time employees joining in the **

In [49]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

repo_id1 = "mistralai/Mistral-7B-Instruct-v0.2"

llm = HuggingFaceEndpoint(
    repo_id=repo_id,
    max_new_tokens=512,
    repetition_penalty=1.03,
    huggingfacehub_api_token=os.environ['HF_TOKEN']
)

model = ChatHuggingFace(llm=llm)

In [52]:
cpt1=ChatPromptTemplate.from_template("answer the question if you are 100 percent sure .or plesae return False ,question:{question}")
repo_id1 = "mistralai/Mistral-7B-Instruct-v0.2" # This line is not needed and causes the error
sto1=StrOutputParser()

main_chain=cpt1|model|sto1

In [53]:
main_chain.invoke({"question":"in my company how may leaves are available"})

'I cannot provide information about the leaves available in your company because I don’t have access to your company’s internal policies, HR systems, or personal data. Please check with your HR department or company policy manual for details on your leave allocation.\n\n*(For privacy and data security reasons, I always return False when asked for personal or company-specific information unless you provide details in the chat.)*\n\nWould you like help finding general guidelines about leaves in labor laws or common company practices? Let me know how I can assist!'

In [54]:
main_chain.invoke({"question":"2+2"})

'The answer is **4**. (2 + 2 = 4) ✅'

In [59]:
def fallback(data):
  answer=main_chain.invoke(data)

  if answer=="False":
    print("now i am calling the retreiver")
    return rag_pipeline.invoke(data["question"])
  else:
    print("i know the answer")
    return answer

In [60]:
fallback_rag=RunnableLambda(fallback)

In [61]:
fallback_rag.invoke({"question":"in my company how may leaves are available"})

i know the answer


'I cannot answer your question about the number of leaves available in your company because:\n\n- **Company policies** (like leave entitlements) are not stored in my knowledge base.\n- I don’t have access to your company’s internal HR systems or leave balance tools.\n\nPlease check with your company’s HR department or leave management platform for the exact details. If you’re unsure how to do this, I’d be happy to guide you on where to look!'

In [62]:
fallback_rag.invoke({"question":"tell me about deep learnini"})

i know the answer


'Your question about **"deep learning"** is very broad, but I can confidently provide a precise and accurate answer based on my knowledge cutoff date of **2023-10-01**. Here’s a detailed response:\n\n---\n\n### **What is Deep Learning?**\n**Deep learning (DL)** is a subset of **machine learning (ML)** that uses **artificial neural networks (ANNs)** with multiple layers (hence "deep") to model and solve complex problems. These networks, inspired by the structure and function of biological brains, can automatically learn hierarchical representations of data through successive layers of transformations, enabling them to handle tasks like pattern recognition, speech synthesis, image classification, and more—often without explicit manual feature engineering.\n\n#### **Key Characteristics:**\n1. **Neural Networks with Many Layers**:\n   Deep learning models (e.g., deep neural networks, CNNs, RNNs, Transformers) consist of **input layers, hidden layers (often dozens or hundreds), and output l